# 03 — Publish **static** Hugging Face Space (free, no PRO)

Gradio Spaces on free `cpu-basic` return **HTTP 402** without a PRO plan.
This notebook uploads ``spaces/ecra-static`` as a **static** Space instead.

- Free for everyone
- HTML portfolio demo + links to the adapter model
- **No live GPU inference** in the Space (run Kaggle / local Gradio for that)

Requires Kaggle secret **`HF_TOKEN`** (write).

In [ ]:
SPACE_REPO_ID = "nuwanda94/earnings-call-research-assistant"
SPACE_DIR = "spaces/ecra-static"
SPACE_SDK = "static"  # do not use gradio on free accounts
SPACE_PRIVATE = False
print(SPACE_REPO_ID, SPACE_SDK)

In [ ]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
if IN_KAGGLE:
    %cd /kaggle/working
    !rm -rf earnings-call-research-assistant
    !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
    REPO = Path("/kaggle/working/earnings-call-research-assistant").resolve()
    %pip install -q huggingface_hub
else:
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO", REPO)
print("static files", list((REPO / SPACE_DIR).iterdir()))

In [ ]:
def _load_hf_token() -> bool:
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        return True
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok and tok.strip():
            os.environ["HF_TOKEN"] = tok.strip()
            os.environ["HUGGING_FACE_HUB_TOKEN"] = tok.strip()
            return True
    return False

assert _load_hf_token(), "Set Kaggle secret HF_TOKEN (write scope)"
print("HF token active: True (value not shown)")

## Push static Space

Creates/updates `https://huggingface.co/spaces/<SPACE_REPO_ID>` with SDK **static**.

In [ ]:
from earnings_call_research_assistant.space_publish import publish_space

plan = publish_space(
    space_dir=SPACE_DIR,
    repo_id=SPACE_REPO_ID,
    private=SPACE_PRIVATE,
    space_sdk=SPACE_SDK,
    commit_message="feat: deploy free static ECRA portfolio Space",
    dry_run=False,
)
print("uploaded:", plan.uploaded)
print("sdk:", plan.space_sdk)
print("Open:", plan.hub_url)
print("notes:", plan.notes)

## Done

Open the Space URL. Live side-by-side generation still runs on Kaggle/local GPU via notebook 02 / `scripts/demo_gradio.py`, not inside this static page.